In [1]:
import math
import torch
import time
import random
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.parameter import Parameter
from torch.nn import init
from torch import Tensor
from scipy.special import gamma 
import matplotlib.pyplot as plt

In [2]:
import torch
from torch.optim import Optimizer


class Lookahead(Optimizer):
    def __init__(self, optimizer, alpha=0.5, k=6):

        if not 0.0 < alpha <= 1.0:
            raise ValueError(f"Invalid alpha: {alpha}")
        if not k >= 1:
            raise ValueError(f"Invalid k: {k}")

        self.optimizer = optimizer
        self.alpha = alpha
        self.k = k
        self.step_counter = 0

        
        self.slow_weights = [p.clone().detach() for group in optimizer.param_groups for p in group['params']]
        for w in self.slow_weights:
            w.requires_grad = False

    def zero_grad(self):
        return self.optimizer.zero_grad()

    def step(self, closure=None):
        loss = self.optimizer.step(closure)  
        self.step_counter += 1

        if self.step_counter % self.k == 0:
            
            idx = 0
            for group in self.optimizer.param_groups:
                for p in group['params']:
                    if p.grad is None:
                        continue
                    slow = self.slow_weights[idx]
                    
                    slow.data.add_(p.data - slow.data, alpha=self.alpha)
                    
                    p.data.copy_(slow.data)
                    idx += 1
        return loss


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Fractional_Order_Matrix_Differential_Solver(torch.autograd.Function):
    @staticmethod
    def forward(ctx,input1,w,b,alpha,k,epoch):
        alpha = torch.tensor(alpha)
        k = torch.tensor(k)
        epoch = torch.tensor(epoch)
        ctx.save_for_backward(input1,w,b,alpha,k,epoch)
        outputs = input1@w + b
        return outputs

    @staticmethod
    def backward(ctx, grad_outputs):
        input1,w,b,alpha,k,epoch = ctx.saved_tensors
        x_fractional, w_fractional = Fractional_Order_Matrix_Differential_Solver.Fractional_Order_Matrix_Differential_Linear(input1,w,b,alpha,k,epoch)   
        x_grad = torch.mm(grad_outputs,x_fractional)
        w_grad = torch.mm(w_fractional,grad_outputs)
        b_grad = grad_outputs.sum(dim=0)
        return x_grad, w_grad, b_grad,None,None,None

    @staticmethod
    def Fractional_Order_Matrix_Differential_Linear(x,w,b,alpha,k,epoch):
        #w
        wf = w[:,0].view(1,-1)
        #main
        w_main = torch.mul(x,(torch.abs(wf)+1e-8)**(1-alpha)/gamma(2-alpha))
        #partial
        x_rows, x_cols = x.size()
        bias = torch.full((x_rows, x_cols),b[0].item())
        bias = bias.to(device)
        w_partial = torch.mul(torch.mm(x,wf.T).view(-1,1).expand(-1,x_cols) - torch.mul(x,wf) + bias, torch.sign(wf)*(torch.abs(wf)+1e-8)**(-alpha)/gamma(1-alpha))
        return w.T, (w_main + torch.exp(-k*epoch)*w_partial).T

class FLinear(nn.Module):
    
    __constants__ = ['in_features', 'out_features']
    in_features: int
    out_features: int
    weight: Tensor

    def __init__(self, in_features: int, out_features: int, alpha=0.9, k = 0.9, bias: bool = True,
                 device=None, dtype=None) -> None:
        factory_kwargs = {'device': device, 'dtype': dtype}
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.k = k

        self.weight = Parameter(torch.empty((in_features, out_features), **factory_kwargs))
        if bias:
            self.bias = Parameter(torch.empty(out_features, **factory_kwargs))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x, epoch):
        return Fractional_Order_Matrix_Differential_Solver.apply(x, self.weight, self.bias, self.alpha, self.k, epoch)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"
    
def split(X,y):
    X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=0.3,shuffle=False)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333,shuffle=False)
    return X_train,X_val,X_test,y_train,y_val,y_test

#Mean Square Error
def MSE(pred,true):
    return np.mean((pred-true)**2)

#Mean Absolute Error
def MAE(pred, true):
    return np.mean(np.abs(pred-true))

def RMSE(pred,true):
    return np.sqrt(np.mean((pred-true)**2))

def MAPE(pred, true):
    return np.mean(np.abs((pred - true) / true))


In [14]:
slide_windows_size = 192  #i.e.,input length 192
pred_length = 384     #i.e.,prediction lengths 384
stock = 'DJI'    #ETTh2,DJI
df_DJIA = pd.read_csv(r'./data/'+stock+'.csv')
# del df_DJIA['date']        #ETT2
del df_DJIA['Date']        #DJI
scaler = MinMaxScaler(feature_range=(0, 1))

sca_DJIA = scaler.fit_transform(df_DJIA)

features_j = 4     #ETTh2:6,DJI:4
def create_sequences(data, slide_windows_size, pred_length):
    X, y = [], []
    for i in range(len(data) - slide_windows_size - pred_length + 1):
        X.append(data[i:i+slide_windows_size, :])  # sliding window size [seq_len, features]
        y.append(data[i+slide_windows_size:i+slide_windows_size+pred_length, features_j])  
    return np.array(X), np.array(y)

X, y = create_sequences(sca_DJIA, slide_windows_size, pred_length)
X = torch.Tensor(X).to(device)
y = torch.Tensor(y).to(device)

X_train,X_val,X_test,y_train,y_val,y_test = split(X,y)   #7:2:1 

In [16]:
alphal = 1.0   
kl = 0.01      #In integer order, k does not play a role.

lrs = [0.01,0.005,0.001]              #ETTh2:[0.01,0.005,0.001]
weight_decays = [0.1,0.01,0.001,0.0001]              

num_feature = 5     #ETTh1:7,DJI:5
batch_size = 256
set_seed()
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size1=256, hidden_size2=128,output_size=pred_length):   
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = FLinear(input_size, hidden_size1, alphal, kl)  
        self.leakrelu1 = nn.LeakyReLU()                          
        self.linear2 = FLinear(hidden_size1, hidden_size2, alphal, kl) 
        self.leakrelu2 = nn.LeakyReLU()
        self.linear3 = FLinear(hidden_size2, output_size, alphal, kl)   

    def forward(self, x, epoch=0):
        x = self.flatten(x)    # (batch_size, seq_len*num_features)
        x = self.leakrelu1(self.linear1(x, epoch)) 
        x = self.leakrelu2(self.linear2(x, epoch))
        x = self.linear3(x, epoch)
        return x

lr_best = 0
weight_decay_best = 0
best_evaluation = 1000000

for lr in lrs:
    for weight_decay in weight_decays:
        set_seed()
        model = MLP(input_size=slide_windows_size*num_feature).to(device)
        num_epochs = 100   #
        best_loss = 1000000
        criterion = nn.MSELoss()
        base_optimizer  = torch.optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)
        optimizer = Lookahead(base_optimizer)
        for ii in range(num_epochs):
            model.train()
            loss_sum = 0
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs,ii)
                loss = criterion(outputs, targets)
                loss_sum += loss
                loss.backward()   #The default value of retain_graph is False.
                optimizer.step()
            # train_loss10.append(loss_sum.cpu().detach().numpy())     ###########
            
            # print(f"Epoch {ii + 1}/{num_epochs}, Train Loss: {loss_sum.cpu().detach().numpy():.4f}")
                
            model.eval()
            with torch.no_grad():
                Val_outputs = model(X_val)
                MSE_val = MSE(y_val.cpu().detach().numpy(),Val_outputs.cpu().detach().numpy())
                
                # val_loss10.append(MSE_val)   ########################Validation_loss
                
                # print(f"Epoch {ii + 1}/{num_epochs}, Val Loss: {MSE_val:.4f}")
                # print('')
                if best_loss > MSE_val:
                    best_loss = MSE_val
                    torch.save(model.state_dict(), r'./model/table_Lookahead/'+stock+'_model_fractional_'+str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth')


        model.load_state_dict(torch.load('./model/table_Lookahead/'+stock+'_model_fractional_'+str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth'))
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
        RMSE10 = RMSE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAE10 = MAE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAPE10 = MAPE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'RMSE:{RMSE10:.4f}')
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAE:{MAE10:.4f}')
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAPE:{MAPE10:.4f}')
        if best_evaluation > RMSE10 + MAE10 + MAPE10:
            best_evaluation = RMSE10 + MAE10 + MAPE10
            print(str(alphal)+';'+str(kl)+';'+str(lr)+';'+str(weight_decay))
            print(f'best_evaluation:{best_evaluation:.4f}')

1.0_0.01_0.01_0.1__RMSE:0.8637
1.0_0.01_0.01_0.1__MAE:0.8591
1.0_0.01_0.01_0.1__MAPE:92.7579
1.0;0.01;0.01;0.1
best_evaluation:94.4807
1.0_0.01_0.01_0.01__RMSE:0.1322
1.0_0.01_0.01_0.01__MAE:0.1138
1.0_0.01_0.01_0.01__MAPE:0.1407
1.0;0.01;0.01;0.01
best_evaluation:0.3867
1.0_0.01_0.01_0.001__RMSE:0.1473
1.0_0.01_0.01_0.001__MAE:0.1274
1.0_0.01_0.01_0.001__MAPE:0.1523
1.0_0.01_0.01_0.0001__RMSE:0.1474
1.0_0.01_0.01_0.0001__MAE:0.1186
1.0_0.01_0.01_0.0001__MAPE:0.1625
1.0_0.01_0.005_0.1__RMSE:0.8645
1.0_0.01_0.005_0.1__MAE:0.8598
1.0_0.01_0.005_0.1__MAPE:86.8996
1.0_0.01_0.005_0.01__RMSE:0.1491
1.0_0.01_0.005_0.01__MAE:0.1241
1.0_0.01_0.005_0.01__MAPE:0.1681
1.0_0.01_0.005_0.001__RMSE:0.1478
1.0_0.01_0.005_0.001__MAE:0.1283
1.0_0.01_0.005_0.001__MAPE:0.1582
1.0_0.01_0.005_0.0001__RMSE:0.1575
1.0_0.01_0.005_0.0001__MAE:0.1284
1.0_0.01_0.005_0.0001__MAPE:0.1822
1.0_0.01_0.001_0.1__RMSE:0.8650
1.0_0.01_0.001_0.1__MAE:0.8604
1.0_0.01_0.001_0.1__MAPE:92.1373
1.0_0.01_0.001_0.01__RMSE:0.2097
1

In [17]:
alphas = [0.9,0.95,0.99,0.999]   
ks = [0.005,0.01,0.05,0.1,0.5,0.9]  

lrs = 0.01
weight_decays =  0.01     

num_feature = 5     #ETTh1:7,DJI:5
batch_size = 256
set_seed()
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size1=256, hidden_size2=128,output_size=pred_length):   
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = FLinear(input_size, hidden_size1, alphal, kl)  
        self.leakrelu1 = nn.LeakyReLU()                          
        self.linear2 = FLinear(hidden_size1, hidden_size2, alphal, kl) 
        self.leakrelu2 = nn.LeakyReLU()
        self.linear3 = FLinear(hidden_size2, output_size, alphal, kl)   

    def forward(self, x, epoch=0):
        x = self.flatten(x)    # (batch_size, seq_len*num_features)
        x = self.leakrelu1(self.linear1(x, epoch)) 
        x = self.leakrelu2(self.linear2(x, epoch))
        x = self.linear3(x, epoch)
        return x

lr_best = 0
weight_decay_best = 0
best_evaluation = 1000000

for alphal in alphas:
    for kl in ks:
        set_seed()
        model = MLP(input_size=slide_windows_size*num_feature).to(device)
        num_epochs = 100   #
        best_loss = 1000000
        criterion = nn.MSELoss()
        base_optimizer  = torch.optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)
        optimizer = Lookahead(base_optimizer)
        for ii in range(num_epochs):
            model.train()
            loss_sum = 0
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs,ii)
                loss = criterion(outputs, targets)
                loss_sum += loss
                loss.backward()   #The default value of retain_graph is False.
                optimizer.step()
            # train_loss10.append(loss_sum.cpu().detach().numpy())     ###########
            
            # print(f"Epoch {ii + 1}/{num_epochs}, Train Loss: {loss_sum.cpu().detach().numpy():.4f}")
                
            model.eval()
            with torch.no_grad():
                Val_outputs = model(X_val)
                MSE_val = MSE(y_val.cpu().detach().numpy(),Val_outputs.cpu().detach().numpy())
                
                # val_loss10.append(MSE_val)   ########################Validation_loss
                
                # print(f"Epoch {ii + 1}/{num_epochs}, Val Loss: {MSE_val:.4f}")
                # print('')
                if best_loss > MSE_val:
                    best_loss = MSE_val
                    torch.save(model.state_dict(), r'./model/table_Lookahead/'+stock+'_model_fractional_'+str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth')


        model.load_state_dict(torch.load('./model/table_Lookahead/'+stock+'_model_fractional_'+str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_.pth'))
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
        RMSE10 = RMSE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAE10 = MAE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        MAPE10 = MAPE(y_test.cpu().numpy(),test_outputs.cpu().detach().numpy())
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'RMSE:{RMSE10:.4f}')
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAE:{MAE10:.4f}')
        print(str(alphal)+'_'+str(kl)+'_'+str(lr)+'_'+str(weight_decay)+'_'+'_'+f'MAPE:{MAPE10:.4f}')
        if best_evaluation > RMSE10 + MAE10 + MAPE10:
            best_evaluation = RMSE10 + MAE10 + MAPE10
            print(str(alphal)+';'+str(kl)+';'+str(lr)+';'+str(weight_decay))
            print(f'best_evaluation:{best_evaluation:.4f}')

0.9_0.005_0.001_0.0001__RMSE:0.1548
0.9_0.005_0.001_0.0001__MAE:0.1316
0.9_0.005_0.001_0.0001__MAPE:0.1597
0.9;0.005;0.001;0.0001
best_evaluation:0.4460
0.9_0.01_0.001_0.0001__RMSE:0.2040
0.9_0.01_0.001_0.0001__MAE:0.1645
0.9_0.01_0.001_0.0001__MAPE:0.2552
0.9_0.05_0.001_0.0001__RMSE:0.1884
0.9_0.05_0.001_0.0001__MAE:0.1497
0.9_0.05_0.001_0.0001__MAPE:0.2323
0.9_0.1_0.001_0.0001__RMSE:0.1909
0.9_0.1_0.001_0.0001__MAE:0.1509
0.9_0.1_0.001_0.0001__MAPE:0.2334
0.9_0.5_0.001_0.0001__RMSE:0.1857
0.9_0.5_0.001_0.0001__MAE:0.1511
0.9_0.5_0.001_0.0001__MAPE:0.2202
0.9_0.9_0.001_0.0001__RMSE:0.1847
0.9_0.9_0.001_0.0001__MAE:0.1506
0.9_0.9_0.001_0.0001__MAPE:0.2182
0.95_0.005_0.001_0.0001__RMSE:0.1827
0.95_0.005_0.001_0.0001__MAE:0.1486
0.95_0.005_0.001_0.0001__MAPE:0.2114
0.95_0.01_0.001_0.0001__RMSE:0.1884
0.95_0.01_0.001_0.0001__MAE:0.1516
0.95_0.01_0.001_0.0001__MAPE:0.2241
0.95_0.05_0.001_0.0001__RMSE:0.1847
0.95_0.05_0.001_0.0001__MAE:0.1494
0.95_0.05_0.001_0.0001__MAPE:0.2185
0.95_0.1_0.0